In [1]:
import pandas as pd
import uuid

In [2]:
knowledge_base = pd.read_parquet(
    "../processed/knowledge_base.parquet"
)

knowledge_base.head()

,document_id,filename,file_type,pages,tenant,category,subcategory,text,metadata
0,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,6,None,None,None,Call Centre Standard Operating Procedures - 20...,{}
1,734bea56-0491-428b-8713-b1be028e6a3d,CIB_CW_09_01_19.pdf,.pdf,7,None,None,None,CIB/CW/09/01/19\nClaims workflow and \ndocumen...,{}
2,4cb2436f-874e-41f5-9653-df17a2a56616,Guardrisk-Insurance-Claims-Management-Framewor...,.pdf,24,None,None,None,Guardrisk Group (Pty) Ltd \nNon-Life Claims M...,{}
3,8ff7ef28-4291-4c3c-81f2-e086e2046687,sample-return-refund-policy-template.pdf,.pdf,3,None,None,None,Sample Return & Refund Policy Template\nReturn...,{}
4,ec941c42-8b7d-4cae-a546-6b441bc1ce24,SOP 2.010 CUSTOMER SERVICE EXPECTATIONS.pdf,.pdf,4,None,None,None,SOP No \n2.010 \nEffective Date \n03.2014 \nRe...,{}


In [3]:
def chunk_text(text,
               chunk_size=500,
               overlap=50):

    words = text.split()

    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):

        chunk = words[i:i+chunk_size]

        chunks.append(
            " ".join(chunk)
        )

    return chunks

In [4]:
sample = knowledge_base.iloc[0]["text"]

chunks = chunk_text(sample)

print(len(chunks))

4


In [5]:
print(chunks[0])

Call Centre Standard Operating Procedures - 2022 Purpose of the Call Centre Why do we have a Call Centre; what is the role of the Call Centre in our customer relationships? The purpose of the Call Centre is to impart information to our internal and external customers, regarding anything related to CPUT, in a professional and friendly manner, thus alleviating queries directed to faculties and departments. Description of Call Centre operations The Call Centre operates from 07:30 – 16:30 from Monday to Thursday; and from 08:00 – 16:00 on Fridays. The Call Centre does not operate on weekends, public holidays and when the Institution is closed. When there is a high volume of emails, overtime is considered, and agents are compensated with time off. The Call Centre services its customers via telephone and e-mail. The Call Centre uses the ITS system and the CPUT website, as well as information received by the faculties and departments to service our customers. Organisational structure of the C

In [6]:
all_chunks = []

for _, row in knowledge_base.iterrows():

    document_chunks = chunk_text(row["text"])

    for chunk_index, chunk in enumerate(document_chunks):

        all_chunks.append({

            "chunk_id": str(uuid.uuid4()),

            "document_id": row["document_id"],

            "filename": row["filename"],

            "file_type": row["file_type"],

            "tenant": row["tenant"],

            "category": row["category"],

            "subcategory": row["subcategory"],

            "chunk_index": chunk_index,

            "chunk_text": chunk

        })

In [7]:
chunks_df = pd.DataFrame(all_chunks)

print(f"Total chunks created: {len(chunks_df)}")

chunks_df.head()

Total chunks created: 48


,chunk_id,document_id,filename,file_type,tenant,category,subcategory,chunk_index,chunk_text
0,b66130c0-3601-446c-abb6-71bae4a21dd0,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,0,Call Centre Standard Operating Procedures - 20...
1,ff395496-839f-449d-be58-ee05f149a7dc,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,1,"peak periods, a staff compliment of at least 1..."
2,5ad64fe9-a281-4222-805a-645b602bdf55,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,2,and abandoned The fortnightly summary historic...
3,041d7c16-2705-4777-b176-5d10ecdb080d,25d2a76d-abf7-4a8b-a494-ce3b082bddaf,Call%20Centre.pdf,.pdf,None,None,None,3,number before transferring. 9. The Agent will ...
4,e4935a1e-5195-42b5-9465-8c3e3aba8a1a,734bea56-0491-428b-8713-b1be028e6a3d,CIB_CW_09_01_19.pdf,.pdf,None,None,None,0,CIB/CW/09/01/19 Claims workflow and documentat...


In [8]:
output_path = "../processed/chunks.parquet"

chunks_df.to_parquet(
    output_path,
    index=False
)

print(f"Saved {len(chunks_df)} chunks to {output_path}")

Saved 48 chunks to ../processed/chunks.parquet
